# 02 — Feature Engineering
RFM calculation, correlation analysis, and churn label creation.

## Setup

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from config import PG_URL
engine = create_engine(PG_URL)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Blues_d')
print("Setup complete")


## 1. RFM feature engineering

In [ ]:

sql = '''SELECT f.customer_id,
DATE '2018-09-01' - MAX(f.order_date)::date AS recency_days,
COUNT(DISTINCT f.order_id) AS frequency,
ROUND(SUM(f.revenue)::numeric,2) AS monetary,
ROUND(AVG(f.review_score)::numeric,2) AS avg_review_score,
SUM(CASE WHEN f.delivery_delay_days>0 THEN 1 ELSE 0 END) AS late_deliveries,
COUNT(DISTINCT f.category) AS unique_categories,
MAX(f.order_date)::date - MIN(f.order_date)::date AS customer_lifetime_days
FROM olist.fact_orders f WHERE f.is_delivered=1 GROUP BY f.customer_id'''
with engine.connect() as conn:
    df = pd.read_sql(text(sql),conn)
df['churned'] = (df['recency_days']>180).astype(int)
print(f"Total customers: {len(df):,}")
print(f"Churned: {df['churned'].sum():,} ({df['churned'].mean()*100:.1f}%)")
print(f"Active:  {(df['churned']==0).sum():,} ({(df['churned']==0).mean()*100:.1f}%)")
df.describe().round(2)


## 2. RFM distributions

In [ ]:

fig,axes = plt.subplots(1,3,figsize=(15,4))
axes[0].hist(df['recency_days'],bins=50,color='steelblue',edgecolor='white')
axes[0].set_title('Recency (days since last order)')
axes[1].hist(df['frequency'].clip(upper=10),bins=10,color='steelblue',edgecolor='white')
axes[1].set_title('Frequency (orders per customer)')
axes[2].hist(df['monetary'].clip(upper=2000),bins=50,color='steelblue',edgecolor='white')
axes[2].set_title('Monetary (total spend $)')
plt.tight_layout(); plt.show()


## 3. Churn vs active customers

In [ ]:

fig,axes = plt.subplots(1,2,figsize=(13,4))
axes[0].pie([df['churned'].sum(),(df['churned']==0).sum()],
            labels=['Churned','Active'],autopct='%1.1f%%',colors=['#e74c3c','#2ecc71'])
axes[0].set_title('Churn vs Active')
for label,grp in df.groupby('churned'):
    axes[1].hist(grp['monetary'].clip(upper=2000),bins=40,alpha=0.6,label='Churned' if label else 'Active')
axes[1].set_title('Spend by Churn Status'); axes[1].legend()
plt.tight_layout(); plt.show()


## 4. Correlation heatmap

In [ ]:

features = ['recency_days','frequency','monetary','avg_review_score','late_deliveries','unique_categories','customer_lifetime_days','churned']
corr = df[features].corr()
plt.figure(figsize=(9,7))
sns.heatmap(corr,annot=True,fmt='.2f',cmap='Blues',center=0,square=True,linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout(); plt.show()


## 5. RFM segmentation

In [ ]:

df['r_score'] = pd.qcut(df['recency_days'],5,labels=[5,4,3,2,1]).astype(int)
df['f_score'] = pd.qcut(df['frequency'].rank(method='first'),5,labels=[1,2,3,4,5]).astype(int)
df['m_score'] = pd.qcut(df['monetary'].rank(method='first'),5,labels=[1,2,3,4,5]).astype(int)
df['rfm'] = df['r_score']+df['f_score']+df['m_score']
def seg(s):
    if s>=13: return 'Champions'
    if s>=10: return 'Loyal'
    if s>=7:  return 'Potential'
    if s>=5:  return 'At Risk'
    return 'Lost'
df['segment'] = df['rfm'].apply(seg)
summary = df.groupby('segment').agg(customers=('customer_id','count'),avg_spend=('monetary','mean'),avg_recency=('recency_days','mean')).round(2).sort_values('avg_spend',ascending=False)
print(summary.to_string())
colors = {'Champions':'#27ae60','Loyal':'#2980b9','Potential':'#f39c12','At Risk':'#e67e22','Lost':'#e74c3c'}
summary['customers'].plot(kind='bar',color=[colors[s] for s in summary.index])
plt.title('Customers per RFM Segment'); plt.ylabel('Customers'); plt.xticks(rotation=0)
plt.tight_layout(); plt.show()
